In [8]:
import urllib.request
import zipfile
import os

# URL of the Dry Bean Dataset from UCI repository
url = 'https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip'
zip_path = 'dry_bean_dataset.zip'

print("Downloading Dry Bean Dataset...")
urllib.request.urlretrieve(url, zip_path)

print("Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dry_bean_temp')

# The dataset zip contains another zip internally: 'DryBeanDataset.zip'
inner_zip = 'dry_bean_temp/DryBeanDataset.zip'
if os.path.exists(inner_zip):
    with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete. 'DryBeanDataset' directory created.")
else:
    # Fallback to copy xlsx directly if layout changes
    for root, dirs, files in os.walk('dry_bean_temp'):
        for file in files:
            if file.endswith('.xlsx') or file.endswith('.csv'):
                os.rename(os.path.join(root, file), file)
    print("Extraction complete.")

# Clean up temp files
if os.path.exists(zip_path): os.remove(zip_path)
import shutil
if os.path.exists('dry_bean_temp'): shutil.rmtree('dry_bean_temp')


Extracting dataset...
Extraction complete.


In [10]:
#!/usr/bin/env python3
"""
Exercise 10: Adaptive Resonance Theory (Fuzzy ART) for Dynamic Category Formation and
             Novel Pattern Discovery using the Dry Bean Dataset

Network  : Fuzzy ART with complement coding.
           input  I = [a, 1 - a]  (a = min-max normalised features in [0, 1])
           choice T_j = |I ^ w_j| / (alpha + |w_j|)
           match      |I ^ w_j| / |I|  >= rho (vigilance)  -> resonance, learn  w_j = beta*(I ^ w_j) + (1-beta)*w_j
           otherwise   category j is reset, the next-best is tried;
           if every category is reset a NEW category is created (w = I).
Labels   : the bean 'Class' column is NEVER used for learning - only to measure purity afterwards.

Usage    : python exercise10_fuzzy_art_beans.py [path_to_dry_bean_file]
           Looks for Dry_Bean_Dataset.xlsx or Dry_Bean_Dataset.csv in the current folder
           (.xlsx needs openpyxl: pip install openpyxl).
Needs    : numpy, pandas
"""
import os
import sys
import numpy as np
import pandas as pd

RNG = np.random.default_rng(11)


# ----------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------
def load_beans(path=None):
    cands = ([path] if path else []) + ["Dry_Bean_Dataset.xlsx", "Dry_Bean_Dataset.csv",
                                        "DryBeanDataset/Dry_Bean_Dataset.xlsx", "dry_bean.csv"]
    f = next((p for p in cands if p and os.path.exists(p)), None)
    if f is None:
        raise SystemExit("Dry Bean dataset not found. Download Dry_Bean_Dataset.xlsx / .csv from\n"
                         "  https://archive.ics.uci.edu/dataset/602/dry+bean+dataset\n"
                         "and put it next to this script, or pass its path as an argument.")
    df = pd.read_excel(f) if f.lower().endswith((".xlsx", ".xls")) else pd.read_csv(f)
    df = df.dropna()
    ccol = next((c for c in df.columns if str(c).strip().lower() == "class"), df.columns[-1])
    names = [str(v) for v in df[ccol]]
    classes = sorted(set(names))
    y = np.array([classes.index(v) for v in names])
    Xdf = df.drop(columns=[ccol]).select_dtypes(include=[np.number])
    return Xdf.values.astype(float), y, classes, list(Xdf.columns), f


def minmax(X):
    lo, hi = X.min(axis=0), X.max(axis=0)
    span = np.where(hi - lo == 0, 1.0, hi - lo)
    return np.clip((X - lo) / span, 0, 1)


def complement(A):
    return np.hstack([A, 1.0 - A])


# ----------------------------------------------------------------------------------------
# Fuzzy ART
# ----------------------------------------------------------------------------------------
class FuzzyART:
    def __init__(self, dim2, n_classes, rho=0.75, alpha=0.001, beta=1.0):
        self.W = np.empty((0, dim2))
        self.lab = np.empty((0, n_classes), int)           # label tallies (analysis only)
        self.size = np.empty(0, int)
        self.rho, self.alpha, self.beta = rho, alpha, beta
        self.dim2, self.n_classes = dim2, n_classes
        self.resets_total = 0

    @property
    def K(self):
        return len(self.W)

    def _scores(self, I):
        m = np.minimum(I, self.W)
        inter = m.sum(axis=1)
        T = inter / (self.alpha + self.W.sum(axis=1))      # choice function
        match = inter / I.sum()                            # match (vigilance) function
        return T, match, m

    def _new(self, I, label):
        self.W = np.vstack([self.W, I])
        row = np.zeros((1, self.n_classes), int)
        if label is not None:
            row[0, label] = 1
        self.lab = np.vstack([self.lab, row])
        self.size = np.append(self.size, 1)
        return self.K - 1

    def present(self, I, label=None, learn=True, rho=None):
        """Returns (category, number_of_resets, created_new). category = -1 for a novel input when learn=False."""
        rho = self.rho if rho is None else rho
        if self.K == 0:
            return (self._new(I, label), 0, True) if learn else (-1, 0, True)
        T, match, m = self._scores(I)
        resets = 0
        for j in np.argsort(-T):
            if match[j] >= rho:                            # resonance
                if learn:
                    self.W[j] = self.beta * m[j] + (1 - self.beta) * self.W[j]
                    self.size[j] += 1
                    if label is not None:
                        self.lab[j, label] += 1
                self.resets_total += resets
                return int(j), resets, False
            resets += 1                                    # reset: vigilance test failed
        self.resets_total += resets
        if learn:
            return self._new(I, label), resets, True
        return -1, resets, True

    def analyze(self, I, top=5):
        T, match, _ = self._scores(I)
        order = np.argsort(-T)
        rows, winner = [], None
        for rank, j in enumerate(order):
            ok = match[j] >= self.rho
            if rank < top:
                rows.append((int(j), T[j], match[j], ok))
            if ok and winner is None:
                winner = int(j)
                if rank >= top:
                    rows.append((int(j), T[j], match[j], ok))
        return rows, winner

    def majority(self):
        return np.where(self.lab.sum(axis=1) > 0, self.lab.argmax(axis=1), -1)


def train(A_idx, C, y, rho, order=None, alpha=0.001, beta=1.0, net=None, checkpoints=None):
    """One incremental pass over the samples (in the given order). Returns net, assignment, history."""
    net = net or FuzzyART(C.shape[1], int(y.max()) + 1, rho, alpha, beta)
    net.rho = rho
    order = np.arange(len(A_idx)) if order is None else order
    assign = np.full(len(A_idx), -1)
    hist = []
    for n, i in enumerate(order, 1):
        j, _, _ = net.present(C[A_idx[i]], int(y[A_idx[i]]))
        assign[i] = j
        if checkpoints and n in checkpoints:
            hist.append((n, net.K))
    return net, assign, hist


def purity(assign, y):
    tot = 0
    for j in np.unique(assign):
        tot += np.bincount(y[assign == j]).max()
    return tot / len(y)


def ari(a, b):
    ua, ia = np.unique(a, return_inverse=True)
    ub, ib = np.unique(b, return_inverse=True)
    cont = np.zeros((len(ua), len(ub)))
    np.add.at(cont, (ia, ib), 1)
    comb = lambda x: x * (x - 1) / 2
    sc, sa, sb, n = comb(cont).sum(), comb(cont.sum(1)).sum(), comb(cont.sum(0)).sum(), comb(len(a))
    exp, mx = sa * sb / n, (sa + sb) / 2
    return (sc - exp) / (mx - exp) if mx != exp else 1.0


# ----------------------------------------------------------------------------------------
# State and helpers
# ----------------------------------------------------------------------------------------
class App:
    pass


app = App()


def ask(prompt, default, cast=int):
    s = input(f"{prompt} [default {default}]: ").strip()
    if not s:
        return default
    try:
        return cast(s)
    except ValueError:
        print("   invalid input, using default.")
        return default


def setup(path=None):
    X, y, classes, feats, f = load_beans(path)
    app.A = minmax(X)
    app.C = complement(app.A)
    app.y, app.classes, app.feats = y, classes, feats
    app.net, app.assign, app.rho = None, None, None
    print(f"Loaded '{f}': {len(y)} beans, {X.shape[1]} features (min-max normalised, complement-coded to "
          f"{app.C.shape[1]} inputs), {len(classes)} bean types: {', '.join(classes)}")


def subsample(n):
    n = min(n, len(app.y))
    return np.sort(RNG.choice(len(app.y), n, replace=False))


def need_net():
    if app.net is None:
        print("   (no trained ART network yet - training with rho = 0.75 on all beans)")
        train_default(0.75)
    return app.net


def train_default(rho):
    idx = np.arange(len(app.y))
    order = RNG.permutation(len(idx))
    cps = {int(v) for v in np.linspace(len(idx) / 10, len(idx), 10)}
    app.net, app.assign, hist = train(idx, app.C, app.y, rho, order, checkpoints=cps)
    app.rho, app.hist, app.idx = rho, hist, idx


# ----------------------------------------------------------------------------------------
# 1. Incremental category formation
# ----------------------------------------------------------------------------------------
def opt1():
    print("\n=== 1. Incremental Bean Category Formation ===")
    rho = ask("Vigilance parameter rho (0-1)", 0.75, float)
    train_default(rho)
    net = app.net
    print(f"\nCategories formed: {net.K}   (beans per category on average: {len(app.y) / net.K:.1f})")
    print("Category growth while beans arrive one at a time:")
    for n, k in app.hist:
        print(f"   after {n:>6} beans: {k:>4} categories  " + "#" * min(k, 60))
    print(f"\nPurity w.r.t. true bean type (not used in training): {purity(app.assign, app.y) * 100:.1f}%")
    print(f"Average resets per bean: {net.resets_total / len(app.y):.2f}")
    print_category_table(net, 10)


def print_category_table(net, top):
    maj = net.majority()
    order = np.argsort(-net.size)[:top]
    print(f"\n{'cat':>5}{'size':>7}  {'majority type':<12}{'purity':>8}")
    for j in order:
        tot = net.lab[j].sum()
        print(f"{j:>5}{net.size[j]:>7}  {app.classes[maj[j]]:<12}{net.lab[j].max() / max(tot, 1) * 100:>7.1f}%")
    if net.K > top:
        print(f"   ... {net.K - top} more categories")


# ----------------------------------------------------------------------------------------
# 2. Vigilance parameter challenge
# ----------------------------------------------------------------------------------------
def opt2():
    print("\n=== 2. Vigilance Parameter Challenge ===")
    n = ask("Beans to use", 4000)
    idx = subsample(n)
    order = RNG.permutation(len(idx))
    print(f"\n{'rho':>6}{'categories':>12}{'purity %':>10}{'avg resets':>12}{'beans/cat':>11}{'singletons':>12}")
    for rho in (0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95):
        net, assign, _ = train(idx, app.C, app.y, rho, order)
        pur = purity(assign, app.y[idx])
        print(f"{rho:>6.2f}{net.K:>12}{pur * 100:>10.1f}{net.resets_total / len(idx):>12.2f}"
              f"{len(idx) / net.K:>11.1f}{int((net.size == 1).sum()):>12}")
    print("\nLow vigilance -> few broad categories (mixed bean types, low purity). High vigilance -> many tight "
          "categories (high purity, but fragmentation and singletons).")


# ----------------------------------------------------------------------------------------
# 3. Resonance and reset analyzer
# ----------------------------------------------------------------------------------------
def opt3():
    print("\n=== 3. Resonance and Reset Analyzer ===")
    net = need_net()
    i = ask(f"Bean index (0-{len(app.y) - 1})", 0)
    i = i if 0 <= i < len(app.y) else 0
    rows, winner = net.analyze(app.C[i], top=6)
    print(f"\nBean {i} (true type {app.classes[app.y[i]]}), vigilance rho = {net.rho}")
    print(f"{'rank':>5}{'category':>10}{'choice T':>11}{'match':>9}   outcome")
    for r, (j, T, m, ok) in enumerate(rows, 1):
        print(f"{r:>5}{j:>10}{T:>11.4f}{m:>9.4f}   " + (f"RESONANCE (match >= {net.rho})" if ok else "RESET (match < rho)"))
    if winner is None:
        print("No category satisfies the vigilance test -> a NEW category would be created.")
    else:
        print(f"Bean joins category {winner} (majority type {app.classes[net.majority()[winner]]}).")
    # dataset-wide statistics
    sample = subsample(1500)
    res = np.array([net.present(app.C[k], learn=False)[1:] for k in sample])
    resets, novel = res[:, 0], res[:, 1].astype(bool)
    print(f"\nOver {len(sample)} beans (learning off): mean resets {resets.mean():.2f}, "
          f"{(resets == 0).mean() * 100:.1f}% resonate with their first choice, "
          f"{novel.mean() * 100:.1f}% would trigger a new category.")
# ----------------------------------------------------------------------------------------
# 4. Novel bean pattern detector
# ----------------------------------------------------------------------------------------
def opt4():
    print("\n=== 4. Novel Bean Pattern Detector ===")
    for k, c in enumerate(app.classes):
        print(f"   {k}. {c} ({int((app.y == k).sum())} beans)")
    h = ask("Hold-out (unseen) bean type number", 2)
    h = h if 0 <= h < len(app.classes) else 2
    rho = ask("Vigilance rho", 0.85, float)
    seen = np.where(app.y != h)[0]
    unseen = np.where(app.y == h)[0]
    RNG.shuffle(seen)
    tr, te_seen = seen[: int(0.7 * len(seen))], seen[int(0.7 * len(seen)):]
    net, _, _ = train(tr, app.C, app.y, rho, np.arange(len(tr)))
    det_seen = np.array([net.present(app.C[i], learn=False)[2] for i in te_seen])
    det_unseen = np.array([net.present(app.C[i], learn=False)[2] for i in unseen])
    best = lambda ids: np.array([net._scores(app.C[i])[1].max() for i in ids[:500]])
    print(f"\nTrained on {len(tr)} beans of {len(app.classes) - 1} types -> {net.K} categories.")
    print(f"Unseen type '{app.classes[h]}': flagged NOVEL for {det_unseen.mean() * 100:.1f}% of {len(unseen)} beans (detection rate)")
    print(f"Known types (held-out beans) flagged novel: {det_seen.mean() * 100:.1f}% of {len(te_seen)} (false alarms)")
    print(f"Mean best-match value: unseen {best(unseen).mean():.3f} vs known {best(te_seen).mean():.3f} (vigilance {rho})")
    print("\nNovel-bean flags by true type:")
    for k, c in enumerate(app.classes):
        ids = unseen if k == h else te_seen[app.y[te_seen] == k]
        if len(ids):
            r = np.mean([net.present(app.C[i], learn=False)[2] for i in ids])
            print(f"   {c:<10} {r * 100:>5.1f}%   {'(unseen)' if k == h else ''}")
    print("A higher vigilance detects more novelty but also raises false alarms on legitimate known beans.")


# ----------------------------------------------------------------------------------------
# 5. Category split and merge explorer
# ----------------------------------------------------------------------------------------
def opt5():
    print("\n=== 5. Category Split and Merge Explorer ===")
    net = need_net()
    maj = net.majority()
    purities = net.lab.max(axis=1) / np.maximum(net.lab.sum(axis=1), 1)
    print("-- SPLIT candidates: large categories mixing several bean types --")
    cands = [j for j in np.argsort(-net.size) if purities[j] < 0.9 and net.size[j] >= 20][:5]
    if not cands:
        print("   none (all sizeable categories are >= 90% pure).")
    for j in cands:
        members = np.where(app.assign == j)[0]
        mix = ", ".join(f"{app.classes[c]} {v}" for c, v in enumerate(net.lab[j]) if v)
        print(f"   category {j}: {net.size[j]} beans, purity {purities[j] * 100:.0f}%  [{mix}]")
        for r2 in (min(app.rho + 0.1, 0.99), min(app.rho + 0.2, 0.99)):
            sub, asg, _ = train(members, app.C, app.y, r2, np.arange(len(members)))
            print(f"      re-learned at rho={r2:.2f}: {sub.K} sub-categories, purity {purity(asg, app.y[members]) * 100:.0f}%")
    print("\n-- MERGE candidates: same-type categories with highly overlapping weights --")
    K = net.K
    sims = []
    sz = net.W.sum(axis=1)
    for a in range(K):
        inter = np.minimum(net.W[a], net.W[a + 1:]).sum(axis=1)
        for off, v in enumerate(inter):
            b = a + 1 + off
            s = v / min(sz[a], sz[b])
            if s >= 0.9 and maj[a] == maj[b] and maj[a] >= 0:
                sims.append((s, a, b))
    sims.sort(reverse=True)
    print(f"   {len(sims)} candidate pairs (similarity >= 0.90, same majority type)")
    for s, a, b in sims[:6]:
        print(f"   categories {a} & {b}: similarity {s:.3f}, sizes {net.size[a]} + {net.size[b]}, type {app.classes[maj[a]]}")
    if sims:
        merged = len({x for _, a, b in sims for x in (a, b)})
        print(f"   merging all candidate groups could remove roughly {max(1, merged // 2)} of {K} categories.")
print("\nSplit = raise vigilance locally; merge = collapse near-duplicate prototypes (lower vigilance).")


# ----------------------------------------------------------------------------------------
# 6. Catastrophic forgetting resistance test
# ----------------------------------------------------------------------------------------
class Softmax:
    """Online softmax regression trained with SGD - baseline that forgets."""
    def __init__(self, d, k, lr=0.1, seed=0):
        self.W = np.zeros((d + 1, k))
        self.lr, self.rng = lr, np.random.default_rng(seed)

    def fit(self, X, y, epochs=30):
        Xa = np.hstack([X, np.ones((len(X), 1))])
        for _ in range(epochs):
            for i in self.rng.permutation(len(X)):
                z = Xa[i] @ self.W
                p = np.exp(z - z.max())
                p /= p.sum()
                p[y[i]] -= 1
                self.W -= self.lr * np.outer(Xa[i], p)

    def predict(self, X):
        return np.argmax(np.hstack([X, np.ones((len(X), 1))]) @ self.W, axis=1)


def art_predict(net, ids):
    maj = net.majority()
    out = []
    for i in ids:
        j, _, _ = net.present(app.C[i], learn=False)
        out.append(maj[j] if j >= 0 else -1)
    return np.array(out)


def opt6():
    print("\n=== 6. Catastrophic Forgetting Resistance Test ===")
    K = len(app.classes)
    for k, c in enumerate(app.classes):
        print(f"   {k}. {c}")
    s = input(f"Task A bean types, comma separated [default 0,1,2]: ").strip() or "0,1,2"
    try:
        A = sorted({int(t) for t in s.split(",") if t.strip() != ""})
        assert A and all(0 <= a < K for a in A) and len(A) < K
    except Exception:
        print("   invalid list, using 0,1,2.")
        A = [0, 1, 2]
    B = [k for k in range(K) if k not in A]
    rho = ask("Vigilance rho", 0.85, float)
    idx = subsample(6000)
    RNG.shuffle(idx)
    cut = int(0.7 * len(idx))
    tr, te = idx[:cut], idx[cut:]
    trA, trB = tr[np.isin(app.y[tr], A)], tr[np.isin(app.y[tr], B)]
    teA, teB = te[np.isin(app.y[te], A)], te[np.isin(app.y[te], B)]
    print(f"\nTask A = {[app.classes[a] for a in A]} ({len(trA)} train / {len(teA)} test)")
    print(f"Task B = {[app.classes[b] for b in B]} ({len(trB)} train / {len(teB)} test)\n")
    # ART
    net, _, _ = train(trA, app.C, app.y, rho, np.arange(len(trA)))
    kA = net.K
    a1 = (art_predict(net, teA) == app.y[teA]).mean() * 100
    train(trB, app.C, app.y, rho, np.arange(len(trB)), net=net)
    a2 = (art_predict(net, teA) == app.y[teA]).mean() * 100
    b2 = (art_predict(net, teB) == app.y[teB]).mean() * 100
    # baseline
    sm = Softmax(app.A.shape[1], K)
    sm.fit(app.A[trA], app.y[trA])
    s1 = (sm.predict(app.A[teA]) == app.y[teA]).mean() * 100
    sm.fit(app.A[trB], app.y[trB])
    s2 = (sm.predict(app.A[teA]) == app.y[teA]).mean() * 100
    sb = (sm.predict(app.A[teB]) == app.y[teB]).mean() * 100
    print(f"{'model':<26}{'Task A after A':>16}{'Task A after B':>16}{'Task B after B':>16}{'forgetting of A':>17}")
    print(f"{'Fuzzy ART (rho=' + str(rho) + ')':<26}{a1:>15.1f}%{a2:>15.1f}%{b2:>15.1f}%{a1 - a2:>16.1f}")
    print(f"{'Softmax SGD (baseline)':<26}{s1:>15.1f}%{s2:>15.1f}%{sb:>15.1f}%{s1 - s2:>16.1f}")
    print(f"\nART grew from {kA} to {net.K} categories while learning Task B; old categories were left untouched "
          "unless a Task B bean genuinely matched them (vigilance protects them).")
    print("The baseline overwrites shared weights, so its Task A accuracy collapses after training on Task B only.")
# ----------------------------------------------------------------------------------------
# 7. Input order sensitivity
# ----------------------------------------------------------------------------------------
def opt7():
    print("\n=== 7. Input Order Sensitivity Challenge ===")
    rho = ask("Vigilance rho", 0.8, float)
    runs = ask("Random orderings", 8)
    idx = subsample(3000)
    y = app.y[idx]
    results, assigns = [], []
    for r in range(runs):
        order = RNG.permutation(len(idx))
        net, asg, _ = train(idx, app.C, app.y, rho, order)
        results.append((f"random #{r + 1}", net.K, purity(asg, y)))
        assigns.append(asg)
    named = {"sorted by bean type": np.argsort(y, kind="stable"),
             "reverse by bean type": np.argsort(-y, kind="stable"),
             "ascending Area feature": np.argsort(app.A[idx][:, 0])}
    for name, order in named.items():
        net, asg, _ = train(idx, app.C, app.y, rho, order)
        results.append((name, net.K, purity(asg, y)))
        assigns.append(asg)
    print(f"\n{'ordering':<24}{'categories':>11}{'purity %':>10}{'ARI vs random #1':>18}")
    for (n, k, p), a in zip(results, assigns):
        print(f"{n:<24}{k:>11}{p * 100:>10.1f}{ari(assigns[0], a):>18.3f}")
    ks = np.array([k for n, k, p in results[:runs]])
    print(f"\nRandom orderings: categories mean {ks.mean():.1f} +/- {ks.std():.1f} (min {ks.min()}, max {ks.max()}).")
    pair = [ari(assigns[a], assigns[b]) for a in range(runs) for b in range(a + 1, runs)]
    if pair:
        print(f"Mean pairwise Adjusted Rand Index between random orderings: {np.mean(pair):.3f} (1.0 = identical partitions)")
    print("ART is order dependent: prototypes depend on which beans arrive first, though purity usually stays similar.")


# ----------------------------------------------------------------------------------------
# 8. Report
# ----------------------------------------------------------------------------------------
def opt8():
    print("\n=== 8. ART Category Formation Report ===")
    net = need_net()
    maj = net.majority()
    print(f"Vigilance rho = {net.rho}, alpha = {net.alpha}, beta = {net.beta} (fast learning), "
          f"inputs = {app.C.shape[1]} (complement coded)")
    print(f"Beans: {len(app.y)}   Categories: {net.K}   Compression: {len(app.y) / net.K:.1f} beans/category")
    print(f"Overall purity: {purity(app.assign, app.y) * 100:.1f}%   Average resets per bean: {net.resets_total / len(app.y):.2f}")
    print(f"Singleton categories: {int((net.size == 1).sum())}   Largest category: {net.size.max()} beans")
    print(f"\n{'bean type':<12}{'beans':>7}{'categories':>12}{'in dominant cat %':>19}")
    for k, c in enumerate(app.classes):
        m = app.y == k
        cats, cnt = np.unique(app.assign[m], return_counts=True)
        print(f"{c:<12}{int(m.sum()):>7}{len(cats):>12}{cnt.max() / m.sum() * 100:>18.1f}%")
    print_category_table(net, 12)
    print("\nCategory prototype ranges for the largest category (original min-max scale 0-1):")
    j = int(np.argmax(net.size))
    d = app.A.shape[1]
    lo, hi = net.W[j, :d], 1 - net.W[j, d:]
    for f, a, b in list(zip(app.feats, lo, hi))[:6]:
        print(f"   {f:<18} [{a:.3f}, {b:.3f}]")
    print("   ...")


MENU = """
============ Fuzzy ART Dry Bean System ============
 1. Incremental Bean Category Formation
 2. Vigilance Parameter Challenge
 3. Resonance and Reset Analyzer
 4. Novel Bean Pattern Detector
 5. Category Split and Merge Explorer
 6. Catastrophic Forgetting Resistance Test
 7. Input Order Sensitivity Challenge
 8. ART Category Formation Report
 9. Exit
===================================================="""
ACTIONS = {"1": opt1, "2": opt2, "3": opt3, "4": opt4, "5": opt5, "6": opt6, "7": opt7, "8": opt8}
def main():
    setup(sys.argv[1] if len(sys.argv) > 1 else None)
    while True:
        print(MENU)
        try:
            c = input("Enter your choice (1-9): ").strip()
        except EOFError:
            break
        if c == "9":
            print("Goodbye!")
            break
        fn = ACTIONS.get(c)
        if fn is None:
            print("Invalid choice, please enter a number from 1 to 9.")
            continue
        fn()


if __name__ == "__main__":
    main()


Split = raise vigilance locally; merge = collapse near-duplicate prototypes (lower vigilance).
Loaded 'Dry_Bean_Dataset.xlsx': 13611 beans, 16 features (min-max normalised, complement-coded to 32 inputs), 7 bean types: BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, SEKER, SIRA

============ Fuzzy ART Dry Bean System ============
 1. Incremental Bean Category Formation
 2. Vigilance Parameter Challenge
 3. Resonance and Reset Analyzer
 4. Novel Bean Pattern Detector
 5. Category Split and Merge Explorer
 6. Catastrophic Forgetting Resistance Test
 7. Input Order Sensitivity Challenge
 8. ART Category Formation Report
 9. Exit
Enter your choice (1-9): 1

=== 1. Incremental Bean Category Formation ===
Vigilance parameter rho (0-1) [default 0.75]: 0.76

Categories formed: 116   (beans per category on average: 117.3)
Category growth while beans arrive one at a time:
   after   1361 beans:   39 categories  #######################################
   after   2722 beans:   58 categories  ###########